In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.functions import (col, count, date_trunc, md5, lit, current_timestamp)
from datetime import datetime

# Widget with default value
dbutils.widgets.text("table_name", "fact_payments")
table_name = dbutils.widgets.get("table_name")

# Configuration & Paths
silver_base = "abfss://processed@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/silver/"
gold_base = "abfss://curated@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/gold/"
target_table = table_name
gold_path = f"{gold_base}{target_table}"

# Silver Sources
df_payments_silver = spark.read.format("delta").load(f"{silver_base}olist_order_payments_dataset")
df_orders_silver = spark.read.format("delta").load(f"{silver_base}olist_orders_dataset")

# Gold Dimension Sources
df_dim_orders = spark.read.format("delta").load(f"{gold_base}dim_orders")
df_dim_customers = spark.read.format("delta").load(f"{gold_base}dim_customers")

# Join
df_enriched = (df_payments_silver.alias("p")
               .join(df_orders_silver.alias("o"), F.col("p.order_id") == F.col("o.order_id"), "left")
               .join(df_dim_orders.alias("do"), F.col("p.order_id") == F.col("do.order_id"), "left")
               .join(df_dim_customers.alias("dc"), F.col("o.customer_id") == F.col("dc.customer_id"), "left"))

df_fact_payments = (df_enriched.select(
                    F.md5(F.concat(
                        F.coalesce(F.col("p.order_id"), lit("UNKNOWN")),
                        F.coalesce(F.col("p.payment_sequential"), lit(0))
                    )).cast("string").alias("payment_key"),
                    F.coalesce(F.col("do.order_key"), lit("-1")).cast("string").alias("order_key"),
                    F.coalesce(F.col("dc.customer_key"), lit("-1")).cast("string").alias("customer_key"),
                    "p.order_id",
                    "p.payment_sequential",
                    "o.order_approved_at",
                    "p.payment_value",
                    "p.payment_installments",
                    "p.payment_type",
                    F.date_trunc("second", F.current_timestamp()).alias("gold_load_at")
))

# Incremental Merge (Upsert)
print(f"--> Starting Gold Load: {target_table}")

if DeltaTable.isDeltaTable(spark, gold_path):
    dt_gold = DeltaTable.forPath(spark, gold_path)

    (dt_gold.alias("target")
     .merge(df_fact_payments.alias("source"), "target.order_id = source.order_id AND target.payment_sequential = source.payment_sequential")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
    
    print(f"--> [Merge] {target_table} updated.")
else:
    #First time load
    df_fact_payments.write.format("delta").mode("overwrite").save(gold_path)
    print(f"--> [INIT] {target_table} created.")

# Audit & Exit
dt_final = DeltaTable.forPath(spark, gold_path)

last_operation = dt_final.history(1).select("operationMetrics").collect()[0][0]
rows_inserted = int(last_operation.get("numTargetRowsInserted", 0))
rows_updated = int(last_operation.get("numTargetRowsUpdated", 0))
total_rows = last_operation.get("numOutputRows", "Check History")

print("-" * 50)
print(f"--> Table: {table_name} | Status: Success")
print(f"--> Rows Processed in last run: {rows_inserted + rows_updated}")
print("-" * 50)

dbutils.notebook.exit(f"Success | Inserted: {rows_inserted} | Updated: {rows_updated}")